#Food Waste Forecasting — CC26-PSU403
**Coding Camp 2026 powered by DBS Foundation**

Model Deep Learning untuk memprediksi **stok optimal** makanan berdasarkan kondisi event.

---

## STEP 1 — Install & Import Library

In [ ]:
!pip install tensorflow scikit-learn pandas numpy matplotlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import pickle
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split

: 

## STEP 2 — Load Dataset

In [ ]:
from google.colab import files
files.upload()

In [ ]:
df = pd.read_csv('food_wastage_cleaned.csv')

print('Shape:', df.shape)
print('\n5 Data Pertama:')
display(df.head())

print('\nMissing Values:')
print(df.isnull().sum())

print('\nStatistik Numerik:')
display(df.describe())

## STEP 3 — Preprocessing: Encoding Kategorikal
ubah kolom teks jadi angka menggunakan supaya bisa diproses model.

In [ ]:
categorical_cols = [
    'Type of Food',
    'Event Type',
    'Storage Conditions',
    'Seasonality',
    'Preparation Method'
]

encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le
    print(f'{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}')

display(df.head())

## STEP 4 — Split Input (X) dan Output (y) + Normalisasi

- **Input (X):** semua kolom kecuali `Quantity of Food`
- **Output (y):** `Quantity of Food` — stok optimal yang mau diprediksi
- **Normalisasi:** MinMaxScaler biar semua nilai ada di rentang 0–1

In [ ]:
X = df.drop(columns=['Quantity of Food'])
y = df['Quantity of Food']

print('Fitur Input (X):', X.columns.tolist())
print('Target Output (y): Quantity of Food')
print(f'Range y: {y.min()} - {y.max()} porsi, rata-rata: {y.mean():.1f} porsi')

# Normalisasi
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Split 80%/20%
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print(f'\nTrain: {X_train.shape} | Test: {X_test.shape}')

## STEP 5 — Custom Components (Layer, Loss, Callback)

Memenuhi syarat MVP Capstone: minimal ada 1 komponen custom. Di sini kita membuat 3 sekaligus:
- `ResidualDenseBlock`: Custom Layer
- `HuberMAELoss`: Custom Loss
- `EarlyStoppingWithRestore`: Custom Callback

In [ ]:
@keras.utils.register_keras_serializable(package="FoodWaste")
class ResidualDenseBlock(keras.layers.Layer):
    def __init__(self, units, dropout_rate=0.3, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.dropout_rate = dropout_rate

    def build(self, input_shape):
        input_dim = input_shape[-1]
        self.dense1 = keras.layers.Dense(self.units, kernel_initializer='he_normal')
        self.bn1 = keras.layers.BatchNormalization()
        self.dense2 = keras.layers.Dense(input_dim, kernel_initializer='he_normal')
        self.bn2 = keras.layers.BatchNormalization()
        self.dropout = keras.layers.Dropout(self.dropout_rate)

    def call(self, inputs, training=False):
        x = self.dense1(inputs)
        x = self.bn1(x, training=training)
        x = tf.nn.relu(x)
        x = self.dropout(x, training=training)
        x = self.dense2(x)
        x = self.bn2(x, training=training)
        return tf.nn.relu(x + inputs)

    def get_config(self):
        config = super().get_config()
        config.update({'units': self.units, 'dropout_rate': self.dropout_rate})
        return config

@keras.utils.register_keras_serializable(package="FoodWaste")
class HuberMAELoss(keras.losses.Loss):
    def __init__(self, delta=1.0, alpha=0.7, **kwargs):
        super().__init__(**kwargs)
        self.delta = delta
        self.alpha = alpha

    def call(self, y_true, y_pred):
        huber = tf.keras.losses.huber(y_true, y_pred, delta=self.delta)
        mae = tf.reduce_mean(tf.abs(y_true - y_pred))
        return self.alpha * huber + (1.0 - self.alpha) * mae

    def get_config(self):
        config = super().get_config()
        config.update({'delta': self.delta, 'alpha': self.alpha})
        return config

class EarlyStoppingWithRestore(keras.callbacks.Callback):
    def __init__(self, patience=20, min_delta=1e-4, verbose=1):
        super().__init__()
        self.patience = patience
        self.min_delta = min_delta
        self.verbose = verbose
        self.wait = 0
        self.best_loss = np.inf
        self.best_weights = None

    def on_train_begin(self, logs=None):
        self.wait = 0
        self.best_loss = np.inf

    def on_epoch_end(self, epoch, logs=None):
        current_loss = logs.get('val_loss')
        if current_loss is None:
            return

        if current_loss < self.best_loss - self.min_delta:
            self.best_loss = current_loss
            self.best_weights = self.model.get_weights()
            self.wait = 0
        else:
            self.wait += 1
            if self.wait >= self.patience:
                self.model.stop_training = True
                if self.verbose > 0:
                    print(f'\nEpoch {epoch+1}: early stopping, restoring best weights.')
                if self.best_weights is not None:
                    self.model.set_weights(self.best_weights)



## STEP 6 — Bangun Arsitektur Model (TensorFlow Functional API)

Arsitektur: `Input(7) → Dense(128) → BatchNorm → Dropout → Dense(64) → BatchNorm → Dropout → Dense(32) → Output(1)`

In [ ]:
inputs = keras.Input(shape=(7,), name='features_input')
x = keras.layers.Dense(128, kernel_initializer='he_normal')(inputs)
x = keras.layers.BatchNormalization()(x)
x = keras.layers.Activation('relu')(x)
x = keras.layers.Dropout(0.2)(x)
x = keras.layers.Dense(256, kernel_initializer='he_normal')(x)
x = keras.layers.BatchNormalization()(x)
x = keras.layers.Activation('relu')(x)

x = ResidualDenseBlock(units=512, dropout_rate=0.3, name='res_block_1')(x)
x = ResidualDenseBlock(units=512, dropout_rate=0.3, name='res_block_2')(x)

x = keras.layers.Dense(64, activation='relu')(x)
x = keras.layers.Dropout(0.2)(x)
x = keras.layers.Dense(32, activation='relu')(x)

outputs = keras.layers.Dense(1, name='quantity_output')(x)

model = keras.Model(inputs=inputs, outputs=outputs, name='FoodStockPredictor')

optimizer = keras.optimizers.Adam(learning_rate=0.001)
model.compile(optimizer=optimizer, loss=HuberMAELoss(delta=1.0, alpha=0.7), metrics=['mae'])
model.summary()



## STEP 7 — Training Model

In [ ]:
callbacks_list = [
    EarlyStoppingWithRestore(patience=20, min_delta=1e-4, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6, verbose=1)
]

history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=200,
    batch_size=32,
    callbacks=callbacks_list,
    verbose=1
)
print('Training selesai')



## STEP 8 — Visualisasi Training History

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot Loss
axes[0].plot(history.history['loss'], label='Train Loss', color='steelblue')
axes[0].plot(history.history['val_loss'], label='Val Loss', color='coral')
axes[0].set_title('Training vs Validation Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (Asymmetric)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot MAE
axes[1].plot(history.history['mae'], label='Train MAE', color='steelblue')
axes[1].plot(history.history['val_mae'], label='Val MAE', color='coral')
axes[1].set_title('Training vs Validation MAE')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE (porsi)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Grafik disimpan: training_history.png')

## STEP 9 — Evaluasi Model di Test Set

In [ ]:
y_pred = model.predict(X_test).flatten()

mae  = np.mean(np.abs(y_test - y_pred))
rmse = np.sqrt(np.mean((y_test - y_pred)**2))
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

print('HASIL EVALUASI (TEST SET)')
print(f'  MAE  : {mae:.2f} porsi')
print(f'  RMSE : {rmse:.2f} porsi')
print(f'  MAPE : {mape:.2f}%')

# Visualisasi Aktual vs Prediksi
plt.figure(figsize=(10, 5))
plt.scatter(y_test, y_pred, alpha=0.5, color='steelblue', label='Prediksi')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
         'r--', label='Ideal (Aktual = Prediksi)')
plt.xlabel('Aktual (porsi)')
plt.ylabel('Prediksi (porsi)')
plt.title('Aktual vs Prediksi Quantity of Food')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('actual_vs_pred.png', dpi=150, bbox_inches='tight')
plt.show()

## STEP 10 — Simpan Model & Artifacts

In [ ]:
import json
import pickle

model.save('food_waste_model.keras')

artifacts = {
    'feature_cols': categorical_cols + ['Number of Guests', 'Wastage Food Amount'],
    'label_encoders': {col: list(le.classes_) for col, le in encoders.items()}
}
with open('model_artifacts.json', 'w') as f:
    json.dump(artifacts, f, indent=2)

with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print('Model dan artifacts disimpan!')



## STEP 11 — Inference Sederhana

Simulasi model dipakai untuk prediksi stok nyata.

Parameters:
- type_of_food        : 'Meat' / 'Vegetables' / 'Fruits' / 'Baked Goods' / 'Dairy Products'
- num_guests          : int (jumlah tamu)
- event_type          : 'Birthday' / 'Corporate' / 'Social Gathering' / 'Wedding'
- storage_conditions  : 'Refrigerated' / 'Room Temperature'
- seasonality         : 'All Seasons' / 'Summer' / 'Winter'
- preparation_method  : 'Buffet' / 'Finger Food' / 'Sit-down Dinner'
- wastage_food_amount : int (estimasi waste historis)

Returns:
- predicted_qty  : int   → stok optimal yang disarankan (porsi)
- waste_ratio    : float → persentase waste terhadap stok prediksi
- recommendation : str   → kategori rekomendasi
- detail         : str   → penjelasan lengkap untuk user

In [ ]:
import json
import pickle

# Jika perlu, load model kembali
# loaded_model = keras.models.load_model('food_waste_model.keras')

with open('model_artifacts.json', 'r') as f:
    artifacts = json.load(f)
loaded_encoders = artifacts['label_encoders']

with open('scaler.pkl', 'rb') as f:
    loaded_scaler = pickle.load(f)

def predict_optimal_stock(type_of_food, num_guests, event_type,
                           storage_conditions, seasonality,
                           preparation_method, wastage_food_amount):
                           
    input_data = {
        'Type of Food': type_of_food,
        'Number of Guests': num_guests,
        'Event Type': event_type,
        'Storage Conditions': storage_conditions,
        'Seasonality': seasonality,
        'Preparation Method': preparation_method,
        'Wastage Food Amount': wastage_food_amount
    }
    
    # Label encoding
    for col in loaded_encoders.keys():
        if input_data[col] in loaded_encoders[col]:
            input_data[col] = loaded_encoders[col].index(input_data[col])
        else:
            input_data[col] = 0
            
    input_df = pd.DataFrame([input_data])
    
    # Pastikan urutan fitur sama persis dengan saat training
    feature_cols = artifacts['feature_cols']
    input_df = input_df[feature_cols]
    
    input_scaled = loaded_scaler.transform(input_df)
    predicted = model.predict(input_scaled, verbose=0)[0][0]
    predicted_qty = max(0, round(predicted))

    waste_ratio = (wastage_food_amount / predicted_qty) * 100 if predicted_qty > 0 else 0

    if waste_ratio > 15:
        recommendation = 'Waste ratio TINGGI — pertimbangkan kurangi order'
        detail = (f'Dari {predicted_qty} porsi yang disiapkan, sekitar {wastage_food_amount} porsi '
                  f'({waste_ratio:.1f}%) diprediksi terbuang. '
                  f'Coba kurangi stok atau ubah metode penyajian.')
    elif waste_ratio < 5:
        recommendation = 'Waste ratio RENDAH — stok sudah sangat efisien'
        detail = (f'Dari {predicted_qty} porsi yang disiapkan, hanya {wastage_food_amount} porsi '
                  f'({waste_ratio:.1f}%) yang diprediksi terbuang. '
                  f'Stok kamu sudah optimal!')
    else:
        recommendation = 'Waste ratio NORMAL — stok sudah efisien'
        detail = (f'Dari {predicted_qty} porsi yang disiapkan, sekitar {wastage_food_amount} porsi '
                  f'({waste_ratio:.1f}%) diprediksi terbuang. '
                  f'Masih dalam batas wajar, bisa tambah buffer 5% jika perlu.')

    return predicted_qty, waste_ratio, recommendation, detail

qty, waste_ratio, rec, detail = predict_optimal_stock(
    type_of_food        = 'Vegetables',
    num_guests          = 450,
    event_type          = 'Birthday',
    storage_conditions  = 'Room Temperature',
    seasonality         = 'Summer',
    preparation_method  = 'Buffet',
    wastage_food_amount = 50
)

print('HASIL PREDIKSI')
print(f'Stok Optimal : {qty} porsi')
print(f'Waste Ratio  : {waste_ratio:.1f}%')
print(f'Rekomendasi  : {rec}')
print(f'Detail       : {detail}')



In [ ]:
# Contoh beberapa skenario sekaligus
print('MULTI-SKENARIO PREDIKSI')

skenarios = [
    ('Vegetables',    450, 'Birthday',         'Room Temperature', 'Summer',      'Buffet',           50),
    ('Meat',          300, 'Wedding',           'Refrigerated',     'Summer',      'Buffet',           25),
    ('Baked Goods',   210, 'Corporate',         'Room Temperature', 'Winter',      'Finger Food',      15),
    ('Dairy Products',350, 'Social Gathering',  'Refrigerated',     'All Seasons', 'Sit-down Dinner',  60),
]
labels = ['Skenario 1', 'Skenario 2', 'Skenario 3', 'Skenario 4']

for label, (tf_, ng, et, sc, se, pm, wfa) in zip(labels, skenarios):
    qty, ratio, rec, detail = predict_optimal_stock(tf_, ng, et, sc, se, pm, wfa)
    print(f'\n🔹 {label}')
    print(f'   Input       : {ng} tamu | {tf_} | {et} | {pm} | waste historis {wfa}')
    print(f'   Stok Optimal: {qty} porsi')
    print(f'   Waste Ratio : {ratio:.1f}%')
    print(f'   Rekomendasi : {rec}')
    print(f'   Detail      : {detail}')